

# Covariance and Correlation
### OPIM 5641 - Business Decision Modeling · Module 4.2
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/7_Nonlinear/2_Covariance_and_Correlation.ipynb)

*Run me top to bottom - **Runtime → Run all**.*

**Read this before the Ms. Womack notebook.** Womack asks how to split money across five
stocks. The answer hinges on one object: the **covariance matrix**. This notebook builds that
object from scratch, and then answers the question every sharp student asks -

> *Correlation is easier to read. Why does the model use covariance instead?*


🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
M4.2 Video 1 - What covariance actually measures

- OPEN: "diversification" is a word people say. Covariance is the number that makes it true.
- Toy data on purpose - 5 days, 2 stocks - so they can do it by hand and check numpy.
- Both stocks rise and fall together -> positive covariance (0.665).
- Then flip stock 2 to the exact opposite -> covariance goes NEGATIVE. Say out loud: this is the
-   pair a portfolio manager wants, because one cushions the other.
- Independent -> covariance near zero.
- DO NOT define correlation yet - that is video 2. Keep this one about co-movement only.
-->


# Part 1 - Covariance

Covariance measures whether two assets move **in the same direction** (positive), **in opposite
directions** (negative), or **independently** (about zero).

$$\text{cov}_{x,y}=\frac{\sum_{i=1}^{N}(x_i-\bar{x})(y_i-\bar{y})}{N-1}$$

Look at the numerator: each day you multiply *how far above its own average* stock x was by
*how far above its own average* stock y was. Both above average, or both below, gives a positive
product. One up while the other is down gives a negative one. Sum them and you have the story
of how the pair moves.

Imagine 5 days of returns:

Day | Stock1 (x) | Stock2 (y)
---|---|---
1 | 1.1% | 3.0%
2 | 1.7% | 4.2%
3 | 2.1% | 4.9%
4 | 1.4% | 4.1%
5 | 0.2% | 2.5%


In [ ]:
import numpy as np

x = [1.1, 1.7, 2.1, 1.4, 0.2]
y = [3.0, 4.2, 4.9, 4.1, 2.5]

print('Stock1 (x):', x, '| mean =', round(np.mean(x), 3))
print('Stock2 (y):', y, '| mean =', round(np.mean(y), 3))

In [ ]:
# np.cov returns the covariance MATRIX, not a single number
np.cov(x, y)

The four entries are:

```
cov(x,x)   cov(x,y)
cov(y,x)   cov(y,y)
```

**Remember:** the **diagonal is each stock's own variance** - `cov(x,x)` is just the variance of x.
Hold on to that; it is the whole answer to the covariance-vs-correlation question later.

And the matrix is symmetric: `cov(x,y) == cov(y,x)`.


In [ ]:
print('cov(x,y) =', np.cov(x, y)[0][1])
print('cov(y,x) =', np.cov(y, x)[0][1])   # same number
print('cov(x,x) =', np.cov(x, y)[0][0], ' <- this is just var(x):', np.var(x, ddof=1))

**Interpreting it:** the covariance here is positive, so the two stocks move together - when
Stock 1 had a good day, so did Stock 2. For a portfolio that is *bad news*: they will crash
together too.

So what does a portfolio manager actually want? Two things that move **opposite** each other.


In [ ]:
# NOTE: new variable names, so x and y above stay intact
y_opposite = [-v for v in x]     # perfectly mirrors x

print('x         :', x)
print('y_opposite:', y_opposite)
print()
print('covariance:', np.cov(x, y_opposite)[0][1], '<- NEGATIVE, as it should be')

**On your own:** build a third series that has nothing to do with `x` (try
`np.random.normal(size=5)`) and check that its covariance with `x` lands near zero. Independent
assets have roughly zero covariance.


🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
M4.2 Video 2 - Correlation - covariance you can actually read

- Covariance has one weakness: the number is unbounded and its size depends on the units.
- Is 0.665 a lot? You cannot say without knowing how volatile the stocks are.
- Correlation fixes that by dividing out both standard deviations -> always between -1 and +1.
- Show corrcoef on the SAME data: covariance 0.665 becomes correlation 0.95. Now you can say
-   'these are nearly locked together' without knowing anything about the scale.
- Land it: correlation is normalized covariance. Same information about direction, plus strength.
- Tee up video 3: so if correlation is easier to read, why does the model not use it?
-->


# Part 2 - Correlation

Covariance told us the **direction**. It is bad at telling us the **strength**, because its size
depends on how volatile the stocks happen to be. Is a covariance of 0.665 big? You cannot answer
that without more context.

Correlation fixes this by dividing out both standard deviations:

$$\text{cor}_{x,y} = \frac{\text{cov}_{x,y}}{\sigma_x \, \sigma_y}$$

That normalization forces the answer into **[-1, +1]**:

- **+1** - move perfectly together
- **0** - unrelated
- **-1** - perfect mirror images


In [ ]:
print('covariance :', np.cov(x, y)[0][1])
print('correlation:', np.corrcoef(x, y)[0][1])

# ...and correlation is just covariance divided by the two standard deviations
manual = np.cov(x, y)[0][1] / (np.std(x, ddof=1) * np.std(y, ddof=1))
print('by hand    :', manual)

**Remember:** correlation is **normalized covariance**. Nothing more. Same direction
information, plus a strength you can compare across any pair of assets on the planet.


🔴
<!-- 🎙 DAVE TALKING POINTS (invisible when rendered - double-click to read):
M4.2 Video 3 - So why does the model eat covariance, not correlation?

- THE question students ask. Answer it properly on the real Womack data.
- Load the same five sectors Womack uses, show .cov() next to .corr().
- Point at the diagonals side by side: covariance diagonal = each stock's OWN variance;
-   correlation diagonal is all 1.0 - it carries no information at all.
- Reason 1 UNITS: risk must come out in return-squared so sqrt(risk) compares to return.
-   Correlation is unitless - a risk number built from it means nothing.
- Reason 2 THE DIAGONAL: build on correlation and you have declared every stock equally risky.
- Reason 3 MAGNITUDE: two utilities at 0.8 and two biotechs at 0.8 are not the same bet.
- Then DEMO it - compute risk both ways on an even split; the correlation version comes out
-   ~200x too big and in nonsense units.
- NOTE the CSV carries a Month column - drop it, or its variance (50.0) swamps everything.
- CLOSE with the line: correlation is for your eyes, covariance is for the optimizer.
-->


# Part 3 - Why the model uses covariance

Here is the portfolio risk that Ms. Womack's model minimizes - a weighted sum over **every pair**
of stocks:

$$\sigma^2_p = \sum_i \sum_j w_i \, w_j \, \text{Cov}(i,j)$$

Since $\text{Cov}(i,j) = \rho_{ij}\,\sigma_i\,\sigma_j$, you *could* write this with correlation
instead - but only by dragging both standard deviations along. Covariance already has them baked
in. Let's see what actually happens if you swap one for the other.

We will use **the same five sectors as the Womack notebook**.


In [ ]:
import pandas as pd

# same returns file the Ms. Womack notebook uses
URL = 'https://drive.google.com/file/d/1Nixf7roe8lfi9U514jOrbqrCRDAHW77o/view?usp=sharing'
fixed_path = 'https://drive.google.com/uc?export=download&id='
df = pd.read_csv(fixed_path + URL.split('/')[-2])

# the file carries a Month index column - drop it, or its variance swamps the matrix
df = df.drop(columns=['Month'])

df.head()

In [ ]:
print('COVARIANCE matrix - what the model uses')
display(df.cov().round(6))

print('CORRELATION matrix - what humans read')
display(df.corr().round(3))

## Look at the two diagonals

That is the whole argument, sitting right there.

- **Covariance diagonal** - each entry is that stock's **own variance**. Computer is roughly
  `0.0096`; Power is roughly `0.0013`. Computer is about **seven times riskier on its own**.
- **Correlation diagonal** - every entry is exactly **1.000**. For every stock. Always.

So if you fed the correlation matrix to the optimizer, you would be telling it that a jumpy tech
stock and a sleepy utility carry *identical* standalone risk. The model would happily load up on
the riskiest name in the list, because you deleted the evidence that it was risky.


In [ ]:
# Ms. Womack's risk formula, computed both ways on an even split
w = np.repeat(1 / len(df.columns), len(df.columns))

risk_cov  = w @ df.cov().values  @ w      # what the model actually does
risk_corr = w @ df.corr().values @ w      # what happens if you swap in correlation

print('risk using COVARIANCE :', round(risk_cov, 8))
print('risk using CORRELATION:', round(risk_corr, 8))
print()
print('ratio                 :', round(risk_corr / risk_cov, 1), 'x too big')

**Caution:** the correlation version is not just *wrong by a scale factor* you could divide out.
It is a different quantity in different units - the covariances have been replaced by numbers that
know nothing about magnitude. Ranking portfolios by it gives you a different, worse answer.

And the reconstruction shows there is no free lunch - to get covariance back out of correlation,
you have to supply exactly the information you threw away:


In [ ]:
sd = df.std()

# Cov(i,j) = Corr(i,j) * sd_i * sd_j
reconstructed = df.corr() * np.outer(sd, sd)

print('max difference vs df.cov():', float((reconstructed - df.cov()).abs().values.max()))
print('(i.e. identical - correlation + the standard deviations == covariance)')

---

## Bottom line

| | Covariance | Correlation |
|---|---|---|
| Range | unbounded | always -1 to +1 |
| Units | return² | none |
| Diagonal | each stock's **own variance** | always 1.0 |
| Good for | **the optimizer** | **your eyes** |

### 🔷 The nugget

**Correlation is for your eyes; covariance is for the optimizer.** Correlation is normalized so a
human can compare any two pairs at a glance. Covariance keeps the scale - and the scale is exactly
what portfolio risk is made of.

**On your own:** find the most *negatively* correlated pair in the correlation matrix above -
**Chemical** is the one pulling against everything else. That pair is doing the most work for
diversification. Then check what the optimizer actually
does with them in the Ms. Womack notebook - does it load up on both?

*Next: [Portfolio Allocation - Ms. Womack](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/7_Nonlinear/Portfolio_Allocation_Womack.ipynb),
where this matrix becomes the risk constraint in a live Pyomo model.*
